In [6]:
import sys
import yaml
import pandas as pd
import os

sys.path.append("../..")

In [14]:
import sys
sys.path.append(r"C:\Users\eduar\Projects\Python\quant_project\src")

In [15]:
from portfolio.tracker import MultiClientPortfolioTracker

In [17]:
import pickle
pkl_path = r"C:\Users\eduar\Projects\Python\quant_project\data\interim\filtered_data.pkl"
with open(pkl_path, "rb") as f:
    filtered_data = pickle.load(f)
pkl_path = r"C:\Users\eduar\Projects\Python\quant_project\data\interim\ef_weights.pkl"
with open(pkl_path, "rb") as f:
    ef_weights = pickle.load(f)

In [ ]:

tracker = MultiClientPortfolioTracker()

# Get current prices from latest filtered data
current_prices = {
        ticker: df["Close"].iloc[-1] for ticker, df in filtered_data.items()
    }
tracker.update_all_market_prices(current_prices)

    # Rebalance to Efficient Frontier weights
if ef_weights:
        for client_name in tracker.get_all_clients():
            trades = tracker.rebalance_client(client_name, ef_weights, current_prices)
            print(f"Trades for {client_name}:", trades)
            # Execute trades
            for ticker, shares in trades.items():
                if shares > 0:
                    tracker.get_client(client_name).input_trade(
                        ticker, shares, current_prices[ticker], "buy"
                    )
                elif shares < 0:
                    tracker.get_client(client_name).input_trade(
                        ticker, -shares, current_prices[ticker], "sell"
                    )

Trades for Eduardo: {'AAPL': np.float64(-0.00446), 'ABT': np.float64(0.002524408703204454), 'BAC': np.float64(0.0), 'C': np.float64(0.018670903904404977), 'DIS': np.float64(-0.008504), 'GOOGL': np.float64(-0.0026717080982190857), 'INTC': np.float64(0.019196896863018443), 'MSFT': np.float64(1.896458211230303e-21), 'WFC': np.float64(6.56194314961155e-18), 'WMT': np.float64(-0.0015761828426520894)}
Trades for Client1: {'AAPL': np.float64(-20.0), 'ABT': np.float64(12.200178307568741), 'BAC': np.float64(0.0), 'C': np.float64(22.547925300722422), 'DIS': np.float64(0.0), 'GOOGL': np.float64(9.491277317115852), 'INTC': np.float64(-76.81685907178328), 'MSFT': np.float64(2.2902585917478805e-18), 'WFC': np.float64(7.924533526741631e-15), 'WMT': np.float64(10.65125606387981)}


C:\Users\eduar\Projects\Python\quant_project\src\portfolio\tracker.py:44: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.history = pd.concat(


AttributeError: 'Portfolio' object has no attribute 'cash'

In [ ]:
    # -------------------------------
    # Calculate portfolio equity curves
    # -------------------------------
    from portfolio.equity_curve import calculate_portfolio_equity_curve

    # Calculate equity curves and metrics
    for client_name in tracker.get_all_clients():
        client = tracker.get_client(client_name)
        client.history_df = calculate_portfolio_equity_curve(client, filtered_data)

        df = client.history_df
        start_value = df["total_value"].iloc[0]
        end_value = df["total_value"].iloc[-1]
        years = (
            pd.to_datetime(df["Date"].iloc[-1]) - pd.to_datetime(df["Date"].iloc[0])
        ).days / 365.25
        cagr = (end_value / start_value) ** (1 / years) - 1
        print(f"{client_name} CAGR: {cagr:.2%}")

    # Save all histories after rebalancing
    tracker.save_all_histories()
    print("Client portfolio histories saved.")